In [3]:
#! pip install prometheus-api-client pandas matplotlib

In [ ]:
import pandas as pd
from prometheus_api_client import PrometheusConnect
import matplotlib.pyplot as plt

# 1. Connect to Prometheus
# Use 'http://localhost:9090' if running locally
# Use 'http://prometheus:9090' if inside the same Docker network
prom = PrometheusConnect(url="http://localhost:9090", disable_ssl=True)


+ In Prometheus, the time duration in brackets—the range vector selector—defines the look-back window that the rate() function uses to calculate the per-second average. So, for `rate(messages_consumed_total[**60m**])`, the **60m** indicates the following:
    - The rate() function calculates how much the counter `(total messages)` increased between the start and the end of that **60-minute** window, then divides that increase by the number of seconds in an hour ($3,600$ seconds).

## Metrics per partition

+ **Avg Msg Sent/per sec (60m)**: Measures the average rate at which the producer is publishing new messages to Kafka over the last hour.

+ **Avg Msg Consumed/per sec (60m) = System Throughput**: Tracks the average throughput of the consumers, showing how many messages are being successfully processed per second.

+ **Consumer Lag (Total)**: The numerical gap between total messages produced and total messages consumed. A rising number here indicates that the consumers are not keeping up with the producer, which is the primary signal that you need to spin up more consumer instances.

+ **Active Consumers**: Displays the current count of unique consumer instances connected to the system, helping verify if our consumer scaling is working as expected.

+ **P50 Latency (sec)**: Indicates the 50th percentile of end-to-end processing time; it means 50% of messages were processed within this duration, highlighting the "average" performance.

+ **P95 Latency (sec)**: Indicates the 95th percentile of end-to-end processing time; it means 95% of messages were processed within this duration, highlighting the "worst-case" performance for most users.

+ **P99 Latency**: The "outlier" experience; essential for finding extreme edge cases.

In [481]:
import pandas as pd

# --- 1. Get the list of Partitions dynamically ---
partition_info = prom.custom_query(query='count(count by (partition) (messages_produced_total))')
num_partitions = int(partition_info[0]['value'][1]) if partition_info else 0
partition_list = list(range(num_partitions))

# --- 2. Define Separate Query Sets ---
# These keys must match so we can pair them in the loop
partition_queries = {
    'Avg Msg Sent/per sec (10m)': 'sum(rate(messages_produced_total[10m])) by (partition)',
    'Avg Msg Consumed/per sec (10m)': 'sum(rate(messages_consumed_total[10m])) by (partition)',
    'Consumer Lag (Total)': 'sum(messages_produced_total) by (partition) - sum(messages_consumed_total) by (partition)',
    'Active Consumers': 'count(messages_consumed_total) by (partition)',
    'P50 Latency last 60m (sec)': 'histogram_quantile(0.50, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le, partition))',
    'P95 Latency last 60m (sec)': 'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le, partition))',
    'P99 Latency last 60m (sec)': 'histogram_quantile(0.99, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le, partition))'
}

all_queries = {
    'Avg Msg Sent/per sec (10m)': 'sum(rate(messages_produced_total[10m]))',
    'Avg Msg Consumed/per sec (10m)': 'sum(rate(messages_consumed_total[10m]))',
    'Consumer Lag (Total)': 'sum(messages_produced_total) - sum(messages_consumed_total)',
    'Active Consumers': 'count(count by (instance) (messages_consumed_total))',
    'P50 Latency last 60m (sec)': 'histogram_quantile(0.50, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le))',
    'P95 Latency last 60m (sec)': 'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le))',
    'P99 Latency last 60m (sec)': 'histogram_quantile(0.99, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le))'
}

rows = []

# --- 3. Fetch Data ---
for label in all_queries.keys():
    metric_row = {'Metric': label}
    
    try:
        # A. Fetch the 'all' value using the specific global query
        res_all = prom.custom_query(query=all_queries[label])
        metric_row['All'] = round(float(res_all[0]['value'][1]), 5) if res_all else 0.0
        
        # B. Fetch the partition-specific values
        res_part = prom.custom_query(query=partition_queries[label])
        partition_map = {
            res['metric'].get('partition', 'unknown'): round(float(res['value'][1]), 5)
            for res in res_part
        }
        
        # C. Map individual partitions to columns
        for p in partition_list:
            val = partition_map.get(str(p), 0.0)
            # Logic: If the metric is Active Consumers, cap the partition value at 1.0
            if label == 'Active Consumers' and val > 1:
                val = 1
            metric_row[f"P_{p}"] = val     
                   
    except Exception as e:
        print(f"Error fetching {label}: {e}")
        metric_row['All'] = "Error"
        for p in partition_list:
            metric_row[f"P_{p}"] = "Error"
            
    rows.append(metric_row)

# --- 4. Create and Display ---
summary_df = pd.DataFrame(rows)

print(f"\n=== System Overview: {num_partitions} Partitions Detected ===")

# Apply rounding and formatting
def format_values(val):
    if isinstance(val, (int, float)):
        # If it's a whole number, show as int, else show 3 decimals
        return f"{int(val)}" if val == int(val) else f"{val:.3f}"
    return val

display(summary_df.style.format(format_values, subset=summary_df.columns.drop('Metric')).hide(axis='index'))
# display(summary_df.style.hide(axis='index').set_properties(**{'text-align': 'center'}))


=== System Overview: 10 Partitions Detected ===


Metric,All,P_0,P_1,P_2,P_3,P_4,P_5,P_6,P_7,P_8,P_9
Avg Msg Sent/per sec (10m),2.696,0.313,0.242,0.262,0.289,0.244,0.292,0.249,0.298,0.244,0.264
Avg Msg Consumed/per sec (10m),2.700,0.306,0.242,0.262,0.289,0.238,0.286,0.248,0.313,0.252,0.264
Consumer Lag (Total),16,9,0,0,0,0,1,0,0,6,0
Active Consumers,3,1,1,1,1,1,1,1,1,1,1
P50 Latency last 60m (sec),3.475,3.900,3.826,3.122,3.023,3.009,3.039,3.368,3.838,3.899,3.693
P95 Latency last 60m (sec),12.726,10.145,9.911,9.686,10.463,9.603,12.131,14.208,14.960,18.353,13.937
P99 Latency last 60m (sec),19.519,18.661,18.221,17.290,18.093,18.414,19.129,18.842,41.018,77.479,19.415


## Metrics per consumer

In [479]:
import pandas as pd

# --- 1. Get the list of Unique Consumers dynamically ---
consumer_info = prom.custom_query(query='count by (consumer_id) (messages_consumed_total)')
consumer_list = [res['metric'].get('consumer_id') for res in consumer_info]
num_consumers = len(consumer_list)

# --- 2. Fetch the Partition Mapping ---
mapping_query = 'sum by (consumer_id, partition) (rate(messages_consumed_total[2m]) > 0)'
mapping_res = prom.custom_query(query=mapping_query)

assignment_map = {}
for res in mapping_res:
    cid = res['metric'].get('consumer_id')
    part = res['metric'].get('partition')
    if cid not in assignment_map:
        assignment_map[cid] = []
    assignment_map[cid].append(part)

# --- 3. Define Metric Queries ---
consumer_queries = {
    'Avg Msg Consumed/per sec (10m)': 'sum(rate(messages_consumed_total[10m])) by (consumer_id)',
    'Msgs Consumed (Last 5m)': 'sum(increase(messages_consumed_total[5m])) by (consumer_id)',
    'P50 Latency last 60m (sec)': 'histogram_quantile(0.50, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le, consumer_id))',
    'P95 Latency last 60m (sec)': 'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le, consumer_id))',
    'P99 Latency last 60m (sec)': 'histogram_quantile(0.99, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le, consumer_id))'
}

all_queries = {
    'Avg Msg Consumed/per sec (10m)': 'sum(rate(messages_consumed_total[10m]))',
    'Msgs Consumed (Last 5m)': 'sum(increase(messages_consumed_total[5m]))',
    'P50 Latency last 60m (sec)': 'histogram_quantile(0.50, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le))',
    'P95 Latency last 60m (sec)': 'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le))',
    'P99 Latency last 60m (sec)': 'histogram_quantile(0.99, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le))'
}

# --- 4. Fetch and Build Rows ---
rows = []

# A. Assigned Partitions Row
partition_row = {'Metric': 'Assigned Partitions', 'All Consumers': 'All'}
for cid in consumer_list:
    display_name = cid if len(cid) < 12 else f"...{cid[-6:]}"
    parts = assignment_map.get(cid, [])
    partition_row[display_name] = ", ".join(sorted(parts, key=int)) if parts else "None"
rows.append(partition_row)

# B. Numerical Metric Rows
for label in all_queries.keys():
    metric_row = {'Metric': label}
    try:
        res_all = prom.custom_query(query=all_queries[label])
        metric_row['All Consumers'] = round(float(res_all[0]['value'][1]), 5) if res_all else 0.0
        
        res_cons = prom.custom_query(query=consumer_queries[label])
        consumer_map = {
            res['metric'].get('consumer_id', 'unknown'): round(float(res['value'][1]), 5)
            for res in res_cons
        }
        
        for cid in consumer_list:
            display_name = cid if len(cid) < 12 else f"...{cid[-6:]}"
            metric_row[display_name] = consumer_map.get(cid, 0.0)
            
    except Exception as e:
        print(f"Error fetching {label}: {e}")
        metric_row['All Consumers'] = "Error"
    rows.append(metric_row)

# --- 5. Format and Display ---
summary_df = pd.DataFrame(rows)

def format_values(val):
    if isinstance(val, (int, float)):
        # Display as integer if no decimal part (ideal for the 5m count)
        return f"{int(val)}" if val == int(val) else f"{val:.3f}"
    return val

display(summary_df.style.format(format_values, subset=summary_df.columns.drop('Metric')).hide(axis='index'))

Metric,All Consumers,...854ba3,...350093,...9b6e06
Assigned Partitions,All,"0, 1, 2, 3","7, 8, 9","4, 5, 6"
Avg Msg Consumed/per sec (10m),2.702,1.193,1.103,0.406
Msgs Consumed (Last 5m),822.792,330.540,260.364,231.888
P50 Latency last 60m (sec),3.498,3.461,3.640,2.161
P95 Latency last 60m (sec),12.865,9.987,15.068,4.976
P99 Latency last 60m (sec),19.547,18.518,32.675,8.918


## Latency graphs

In [246]:
import plotly.express as px
from datetime import datetime, timedelta
minutes_interval = 10.0
time_bucket_for_average = 5 #minutes

# 2. Setup Time Range (Last 12 hours)
end_time = datetime.now()
start_time = end_time - timedelta(hours=12)
step = "1m"

query = f'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[{time_bucket_for_average}m])) by (le, partition))'

# 3. Fetch Data
result = prom.custom_query_range(
    query=query,
    start_time=start_time,
    end_time=end_time,
    step=step,
)

# 4. Flatten the result into a DataFrame
df_list = []
for series in result:
    partition = series['metric'].get('partition', 'all')
    for val in series['values']:
        df_list.append({
            'timestamp': pd.to_datetime(val[0], unit='s'),
            'partition': f"Partition {partition}",
            'latency_sec': float(val[1])
        })

df = pd.DataFrame(df_list)
df['timestamp'] = pd.to_datetime(df['timestamp']) 

# 5. Create Plotly Graph
fig = px.line(
    df, 
    x='timestamp', 
    y='latency_sec', 
    color='partition',
    title='P95 End-to-End Latency (Interactive)',
    labels={'timestamp': 'Time', 'latency_sec': 'Latency (Seconds)'},
    template='plotly_white'
)

# Convert to milliseconds for Plotly
ms_interval = minutes_interval * 60 * 1000
# Logic to determine the best format
if minutes_interval < 60:
    # Under 1 hour: Show minutes and seconds
    custom_format = "%H:%M:%S"
elif minutes_interval < 1440: # 24 hours
    # 1 to 24 hours: Show Hour:Minute and Short Date
    custom_format = "%H:%M\n%b %d"
else:
    # Over 24 hours: Focus on the Date
    custom_format = "%b %d\n%Y"

fig.update_xaxes(
    type='date',             # Force the axis to recognize dates
    tickmode='linear',
    tick0=df['timestamp'].min(), 
    dtick=ms_interval,
    tickformat=custom_format,
    ticks="outside",
    ticklen=10,
    tickcolor='black',
    showgrid=True,
    gridcolor='LightPink'
)
fig.show()

# 6. Display Stats Table
print("--- Performance Stats ---")
stats_table = df.groupby('partition')['latency_sec'].agg(['mean', 'max', 'std']).reset_index()
display(stats_table)

--- Performance Stats ---


,partition,mean,max,std
0,Partition all,103.3393,178.333333,55.389542


In [ ]:
import pandas as pd
from prometheus_api_client import PrometheusConnect
import matplotlib.pyplot as plt

# 1. Connect to Prometheus
# Use 'http://localhost:9090' if running locally
# Use 'http://prometheus:9090' if inside the same Docker network
prom = PrometheusConnect(url="http://localhost:9090", disable_ssl=True)

# 2. Define the Query (P95 Latency)
query = 'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[5m])) by (le))'

# 3. Fetch Data as a list of dicts and convert to DataFrame
result = prom.custom_query(query=query)
df_list = []

for series in result:
    partition = series['metric'].get('partition', 'all')
    value = float(series['value'][1])
    timestamp = pd.to_datetime(series['value'][0], unit='s')
    df_list.append({'timestamp': timestamp, 'partition': partition, 'latency_sec': value})

df = pd.DataFrame(df_list)

# 4. Display the Table of Stats
print("--- Latency Statistics per Partition ---")
stats_table = df.groupby('partition')['latency_sec'].agg(['mean', 'max', 'min', 'std']).reset_index()
display(stats_table)

# 5. Simple Plot
df.pivot(index='timestamp', columns='partition', values='latency_sec').plot(kind='line', figsize=(10, 5))
plt.title("P95 Latency over Time")
plt.ylabel("Seconds")
plt.show()

In [ ]:
metrics = {
    "Total End-to-End": "message_processing_latency_seconds_bucket",
    "Consumer Queue Time": "message_queue_time_seconds_bucket"
}
quantiles = [0.50, 0.95, 0.99]
results = []

for name, metric in metrics.items():
    row = {"Stage": name}
    for q in quantiles:
        query = f'histogram_quantile({q}, sum(rate({metric}[60m])) by (le))'
        res = prom.custom_query(query)
        val = float(res[0]['value'][1]) if res else 0
        row[f"P{int(q*100)}"] = round(val, 4)
    results.append(row)

df = pd.DataFrame(results).set_index("Stage")

# Calculate Internal
internal = (df.loc["Total End-to-End"] - df.loc["Consumer Queue Time"]).clip(lower=0)
internal.name = "Producer Internal"

# Combine and reorder
final_df = pd.concat([df, pd.DataFrame([internal])])
# This line ensures 'Total' is always at the bottom
final_df = final_df.reindex(["Producer Internal", "Consumer Queue Time", "Total End-to-End"]).reset_index()

print("Latency Summary (Seconds):")
display(final_df)